<a href="https://colab.research.google.com/github/miguelgorr/Projects/blob/main/U87_MS_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd, numpy as np, os, requests, collections
from scipy import stats

In [ ]:
#Identify the Characteristic Processes at each time interval

In [2]:
U87_xlsm_files = [
                  'U87_D12_R3.xlsm',
                  'U87_D3_R4.xlsm',
                  'U87_D6_R4.xlsm',
                  'U87_D9_R1.xlsm'
                  ]

In [63]:
#Convert .xlsm into pandas df and organize in dict with sample name as key and pandas df as value

mydrive_path = '/content/drive/MyDrive/Gorriceta Research 2026/Reeck Lab/U87_MS_Analysis/U87_Raw_Mass_Spec/'
U87_df_dict = {}

for file in U87_xlsm_files:

  file_path = f'{mydrive_path}{file}'
  df = pd.read_excel(file_path)
  df_indexed_gene = df.set_index('Gene Symbol')
  U87_df_dict[file] = df_indexed_gene
  print(f'Successfully uploaded {file}')

Successfully uploaded U87_D12_R3.xlsm
Successfully uploaded U87_D3_R4.xlsm
Successfully uploaded U87_D6_R4.xlsm
Successfully uploaded U87_D9_R1.xlsm


In [85]:
class MassSpec:

  def __init__(self, name, data):
    self.name = name
    self.__data = data

#View-only df
  def view_data(self):
    return self.__data

#Search for a specific gene symbol
  @property
  def loc(self):
    return self.__data.loc

#View colnames
  @property

  def columns(self):
    return self.__data.columns

#Index a part of the df without overwriting
  def __getitem__(self, index: int | slice | str) -> pd.DataFrame | pd.Series :
    return self.__data[index]

  def __filterby__(self, column: str, value: int | float, operator: str = '>' ) -> pd.DataFrame:
    if operator == '>':
      return self.__data[self.__data[column] > value]

    elif operator == '<':
      return self.__data[self.__data[column] < value]

    elif operator == '>=':
      return self.__data[self.__data[column] >= value]

    elif operator == '<=':
      return self.__data[self.__data[column] <= value]

    else:
      raise ValueError('Please input a valid operator Ex: < or >')

In [86]:
MS_class_dict = {}

#Creates a dictionary where key = filename and value is the MassSpec object
for key, value in U87_df_dict.items():
   MS_class_dict[key]= MassSpec(key, value)



In [6]:
#Use this to see what the column names are for the dfs
def view_colnames(dict):
  for key,item in dict.items():
    if isinstance(item, pd.DataFrame):
      print(item.columns)
    else:
      print(f'{key} is not a pandas DataFrame object')

In [ ]:
#Create filtering criteria

#Find the proportion of the total PSMs that each protein accounts for
def majority_psms(dict_of_samples):
  for key,item in dict_of_samples.items():
    list_of_proportions = []
    df_length = len(item)
    psms_series = item['# PSMs']
    for entry in psms_series:
      proportion_of_total = entry / df_length
      item['Proportion Total PSMs'] = proportion_of_total

#High PSMs with low coverage
def high_psms_low_cov(dict_of_samples):
  cov_psms_dict = {}
  for key, item in dict_of_samples.items():
    #Check if you have a dictionary of pd DataFrames and they have the right columns
    if isinstance(item, pd.DataFrame) and all(col in item.columns for col in ['Gene Symbol', 'Coverage [%]', '# PSMs']):
      cov_psms_df = item[['Gene Symbol','Coverage [%]', '# PSMs']]


      #Check for coverage <50%
      cov_psms_df = cov_psms_df[~(cov_psms_df['Coverage [%]'] > 50)]

      #Filter by PSMs > trimmed mean (trimming the top and bottom 5%)
      trimmed_mean_val = stats.trim_mean(cov_psms_df['# PSMs'], proportiontocut=0.10)
      print(f'Trimmed Mean: {trimmed_mean_val}')
      cov_psms_df = cov_psms_df[~(cov_psms_df['# PSMs'] < trimmed_mean_val)]

      #Sort by # PSMs highest to lowest
      cov_psms_df = cov_psms_df.sort_values(by='# PSMs', ascending=False)


      cov_psms_dict[key] = cov_psms_df

      #Save to Google Drive Folder
  output_path = '/content/drive/MyDrive/Gorriceta Research 2026/Reeck Lab/U87_MS_Analysis/U87_Analysis_Results'
  os.makedirs(output_path, exist_ok=True)

  for filename, df in cov_psms_dict.items():
    clean_name = filename.removesuffix('.xlsm')
    save_file_path = f'{output_path}/{clean_name}_filtered.xlsx'
    df.to_excel(save_file_path, index=False)


  return cov_psms_dict




In [ ]:
#Check for max PSM for each day
top_10_dict = {}
gene_id_dict = {}

for df_name, df in U87_df_dict.items():
  sorted_psm_df = df.sort_values(by='# PSMs', ascending=False)
  top_10_df = sorted_psm_df.iloc[:10]
  top_10_dict[df_name] = top_10_df


  #Get a dict of the Gene IDs
  gene_id_dict[df_name] = top_10_df[['Gene Symbol', 'Reactome Pathways']]
